# Question 2 — Applied: MLflow Experiment Comparison

## Setup

In [1]:
import sys
print(sys.executable)

/home/kevin/AIops_lab/AIOps-A1/.venv/bin/python


In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp")
print("Tracking URI:", mlflow.get_tracking_uri())

2026/08/30 19:17:14 INFO mlflow.tracking.fluent: Experiment with name 'mnist-mlp' does not exist. Creating a new experiment.


Tracking URI: http://localhost:5000


## Starter script

Loading mnist instead of iris and an mlp model instead of random forest

In [3]:
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

X = X / 255.0
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def train_and_evaluate(hidden_layer_sizes=(128,), alpha=1e-4,
                       learning_rate_init=1e-3, max_iter=100):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        alpha=alpha,
        learning_rate_init=learning_rate_init,
        max_iter=max_iter,
        early_stopping=True,
        n_iter_no_change=5,
        random_state=42,
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    return model, acc, f1

_, acc, f1 = train_and_evaluate()
print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}")

accuracy=0.9774  f1_macro=0.9773


## Logging

In [4]:
def train_and_log(hidden_layer_sizes=(128,), alpha=1e-4,
                  learning_rate_init=1e-3, max_iter=100, run_name=None):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("alpha", alpha)
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("width", hidden_layer_sizes[0])
        mlflow.log_param("depth", len(hidden_layer_sizes))

        model, acc, f1 = train_and_evaluate(
            hidden_layer_sizes, alpha, learning_rate_init, max_iter
        )

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)
        mlflow.log_metric("n_iter_actual", model.n_iter_)

        # the loss curve — one point per epoch
        for epoch, loss in enumerate(model.loss_curve_):
            mlflow.log_metric("train_loss", loss, step=epoch)
        if hasattr(model, "validation_scores_"):
            for epoch, score in enumerate(model.validation_scores_):
                mlflow.log_metric("val_accuracy", score, step=epoch)

        mlflow.set_tag("team", "data-science")
        mlflow.sklearn.log_model(model, name="model",
    skops_trusted_types=["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"],
)

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id}  |  acc={acc:.4f}  f1={f1:.4f}")
        return run_id

## Sweep (12 experiments)

In [5]:
from itertools import product

learning_rates = [1e-4, 1e-3, 1e-2]
widths = [64, 128]
depths = [1, 2]

combos = list(product(learning_rates, widths, depths))
sweep_run_ids = []

for i, (lr, width, depth) in enumerate(combos, 1):
    print(f"[{i}/{len(combos)}] lr={lr:g} w={width} d={depth}")
    hidden = (width,) * depth
    rid = train_and_log(
        hidden_layer_sizes=hidden,
        learning_rate_init=lr,
        run_name=f"mlp-lr{lr:g}-w{width}-d{depth}",
    )
    sweep_run_ids.append(rid)

print(f"{len(sweep_run_ids)} runs completed")

[1/12] lr=0.0001 w=64 d=1


/home/kevin/AIops_lab/AIOps-A1/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run cc7222626a8e4a509c996e6eed2e0901  |  acc=0.9689  f1=0.9686
🏃 View run mlp-lr0.0001-w64-d1 at: http://localhost:5000/#/experiments/1/runs/cc7222626a8e4a509c996e6eed2e0901
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2/12] lr=0.0001 w=64 d=2
Logged run 72f5a277e143461f946027739219f321  |  acc=0.9669  f1=0.9665
🏃 View run mlp-lr0.0001-w64-d2 at: http://localhost:5000/#/experiments/1/runs/72f5a277e143461f946027739219f321
🧪 View experiment at: http://localhost:5000/#/experiments/1
[3/12] lr=0.0001 w=128 d=1


/home/kevin/AIops_lab/AIOps-A1/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run e4cbcbd33ff4409180efeff316cbefa9  |  acc=0.9749  f1=0.9747
🏃 View run mlp-lr0.0001-w128-d1 at: http://localhost:5000/#/experiments/1/runs/e4cbcbd33ff4409180efeff316cbefa9
🧪 View experiment at: http://localhost:5000/#/experiments/1
[4/12] lr=0.0001 w=128 d=2
Logged run 2c46d20aa2694c1e91b69d3397ba13d1  |  acc=0.9741  f1=0.9739
🏃 View run mlp-lr0.0001-w128-d2 at: http://localhost:5000/#/experiments/1/runs/2c46d20aa2694c1e91b69d3397ba13d1
🧪 View experiment at: http://localhost:5000/#/experiments/1
[5/12] lr=0.001 w=64 d=1
Logged run 65fa1eeb0d61420eb3a67664f6dbda2c  |  acc=0.9731  f1=0.9729
🏃 View run mlp-lr0.001-w64-d1 at: http://localhost:5000/#/experiments/1/runs/65fa1eeb0d61420eb3a67664f6dbda2c
🧪 View experiment at: http://localhost:5000/#/experiments/1
[6/12] lr=0.001 w=64 d=2
Logged run 17c9551ff9c94205bd3edfff5849b65c  |  acc=0.9714  f1=0.9712
🏃 View run mlp-lr0.001-w64-d2 at: http://localhost:5000/#/experiments/1/runs/17c9551ff9c94205bd3edfff5849b65c
🧪 View experiment a

## Finding best run

In [6]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp"],
    order_by=["metrics.accuracy DESC"],
)

cols = ["params.learning_rate_init", "params.width", "params.depth",
        "metrics.accuracy", "metrics.f1_macro", "metrics.n_iter_actual"]
print(runs_df[cols].head(20).to_string(index=False))

params.learning_rate_init params.width params.depth  metrics.accuracy  metrics.f1_macro  metrics.n_iter_actual
                    0.001          128            1          0.977429          0.977302                   19.0
                    0.001          128            2          0.977143          0.976986                   16.0
                   0.0001          128            1          0.974929          0.974748                  100.0
                     0.01          128            1          0.974429          0.974273                   24.0
                   0.0001          128            2          0.974143          0.973921                   59.0
                    0.001           64            1          0.973143          0.972915                   33.0
                    0.001           64            2          0.971429          0.971237                   19.0
                     0.01          128            2          0.971357          0.970962                   16.0
 